In [ ]:
%pip install -q -U alphagenome

In [ ]:
from google.colab import userdata, files
from alphagenome.atlas import atlas
from alphagenome.data import genome
from pathlib import Path
import zipfile

client = atlas.create(
    api_key=userdata.get("ALPHA_GENOME_API_KEY")
)

# 示例：查询人类 SCN1A P1a 所有的突变，
# 获取这些突变对 SCN1A 的 RNA 表达效应。
result = client.query_interval(
    #interval=genome.Interval("chr2", 166148180, 166151550), #P1a
    interval=genome.Interval("chr2", 166127360, 166129030), #P1b
    #interval=genome.Interval("chr2", 166077140, 166079490), #P1c
    requested_scorers=["RNA_SEQ"],
    gene_names=["SCN1A"],
)

rna = result["RNA_SEQ"]

# 保存分数及其对应的基因、突变和实验 track 信息。
out = Path("/content/SCN1A_RNA")
out.mkdir(exist_ok=True)

rna.to_df().to_csv(out / "raw_scores.csv")
rna.obs.to_csv(out / "variants_and_genes.csv")
rna.var.to_csv(out / "tracks.csv")

if "quantiles" in rna.layers:
    rna.to_df(layer="quantiles").to_csv(out / "quantile_scores.csv")

display(rna.obs.head())
display(rna.to_df().head())

# 打包并下载结果到电脑。
zip_path = "/content/SCN1A_RNA.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in out.glob("*.csv"):
        archive.write(path, arcname=path.name)

files.download(zip_path)